## CSC 380 HW#4: LLM and RAG -- **Starter Code**

Spring 2026 HW#4 — fill in every `TODO` block and run all cells.


## Write your name here.

In [ ]:
# Setup  (do not modify)
# ----------------------------
# Colab already has torch, transformers, and sentence-transformers pre-installed.
# Only install the LangChain ecosystem packages that are not pre-installed.
# Do NOT use -U or --force-reinstall here — that breaks Colab's torch/transformers.
!pip install -q \
    langchain langchain-core langchain-community \
    langchain-huggingface langchain-openai langchain-text-splitters \
    faiss-cpu openai tiktoken pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# Imports  (do not modify)
# ----------------------------
import os, re, json as _json, time
import requests
from typing import List, Dict, Tuple
from IPython.display import Markdown, display
import numpy as np

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# Load API keys from CoLab secret keys  (do not modify)
from google.colab import userdata

langchain_api = userdata.get('langchain_api')
openai_api    = userdata.get('openai_api')
hf_token      = userdata.get('HF_TOKEN')

os.environ['OPENAI_API_KEY']           = openai_api
os.environ['LANGCHAIN_API_KEY']        = langchain_api
os.environ['HUGGINGFACEHUB_API_TOKEN'] = hf_token

## Part 0: Define Core Variables and Functions

In [ ]:
# First declare some constants

# models
embedding_model = "all-MiniLM-L6-v2"
rag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_llm = ChatOpenAI(model="gpt-4o", temperature=0)


# TODO (1): Write 8 test questions where 4 of them are answerable and another
#           4 are unanswerable.  Read the assignment page for more specifications.
#test_queries = [] # you fill in here
test_queries = [
    # write your 8 questions here
]


## Part 1: Define Core Functions

In [ ]:
# @title
# 1-1: Load documents  (provided — do not modify)
# ----------------------------
def load_documents() -> List[Document]:
    """
    Load the nutrition / diabetes documents from GitHub.
    Returns a list of Document objects.
    """
    url = "https://raw.githubusercontent.com/ntomuro/CSC380/main/HW4-LLM_RAG/data"
    document_filenames = [
        "Human-Nutrition-2020-Edition-1598491699.txt",
        "dci190009_pdf.txt",
        "dci190014_pdf.txt"
    ]
    documents = []
    for doc in document_filenames:
        document_url = f"{url}/{doc}"
        try:
            response = requests.get(document_url)
            response.raise_for_status()
            documents.append(Document(page_content=response.text,
                                      metadata={"source": document_url}))
        except requests.exceptions.RequestException as e:
            print(f"Error loading {document_url}: {e}")
    return documents


# 1-2: Preprocess documents
# ----------------------------
def preprocess_documents(documents: List[Document]) -> List[Document]:
    """
    Split documents into smaller chunks for vector storage.
    Returns a list of Document chunks.
    """
    # TODO (1): Create a text splitter and split the documents into chunks.
    #
    # Hint: Use RecursiveCharacterTextSplitter(chunk_size=..., chunk_overlap=...)
    # Hint: Call text_splitter.split_documents(documents) to get the chunks.
    pass  # replace with your code


# 1-3: Vector Store Population
# ----------------------------
def create_vector_store(documents: List[Document], embedding_model_name: str):
    """
    Build a FAISS vector store from document chunks and return a retriever.
    FAISS is in-memory — no disk writes, no permission issues.
    Returns a retriever object.
    """
    # TODO (2): Implement vector store creation in two steps.
    # Step 1 — Create an embedding function:
    # Step 2 — Build the FAISS vector store and return a retriever:
    pass  # replace with your code


# 1-4: The RAG Chain  (modern LCEL style)
# ----------------------------
def format_docs(docs):   # helper — joins chunk text with blank lines
        return "\n\n".join(d.page_content for d in docs)

def run_rag_query(query: str, retriever) -> Tuple[str, List[Document]]:
    """
    Execute a RAG query and return (answer, retrieved_docs).

    The pipeline follows the LCEL (LangChain Expression Language) pattern:
        inputs -> prompt -> LLM -> string output
    """
    # Step 1 — Retrieve relevant chunks.
    # TODO  (4): Invoke the retriever with the query to get a list of Document objects.
    #            Name the retrieved doc as 'retrieved_docs'.


    # Step 2 — Define a prompt template.
    # TODO (5): Use PromptTemplate.from_template(...) to create a RAG prompt that:
    #   • includes a {context} variable (the retrieved text) and a {question} variable.
    #   • instructs the model to answer from the context only and say clearly
    #     "I don't know" or something if the answer isn't there.



    # Step 3 — Build the LCEL chain with | (pipe) operators.
    # TODO (6): Assemble the chain -- process pipeline, consisting of
    #  1. concatenated retrieved docs (using the helper function 'format_docs()')
    #  2. RAG prompt
    #  3. LLM (the RAG model)
    #  4. output parser

    # Step 4 — Run the chain.
    # TODO (7): Invoke the chain with query and get the answer string.
    answer = chain.invoke(query)

    return answer, retrieved_docs


# 1-6: LLM-as-Judge Evaluation  (agentic LLM-based evaluation)
# ----------------------------
from langchain_core.output_parsers import JsonOutputParser

def evaluate_results(query: str, retrieved_docs: List[Document], answer: str) -> Tuple[int, int]:
    """
    Use the judge LLM call to evaluate the RAG system's output.
    Returns (retrieval_score, answer_score) each on a 1-5 scale.

    This is an agentic pattern: instead of a human scoring every response,
    an LLM judge critiques the system automatically.

    Scores:
      retrieval_score: 1=irrelevant,.. YOU DECIDE (up to 5=..)
      answer_score:    1=incorrect,  YOU DECIDE (up to 5=..)
    """
    # TODO (8): Run an LLM-as-judge chain in five steps.
    #
    # Step 1 — Write an eval_prompt using PromptTemplate.from_template(...).
    #   Include variables: {query}, {context}, {answer}.
    #   Ask the LLM to score retrieval (1-5) and answer quality (1-5).
    #   Tell it to respond with ONLY this JSON and nothing else:
    #     {"retrieval_score": <int>, "answer_score": <int>, "reason": "<one sentence>"}
    eval_prompt = PromptTemplate.from_template(
        # (**) YOU START THE PROMPT
        #
        #
        "Respond with ONLY valid JSON, nothing else:\n"
        '{{"retrieval_score": <int>, "answer_score": <int>, "reason": "<one sentence>"}}'
    )

    # Step 2 — Build the chain ('judge chain'):
    judge_chain = eval_prompt | judge_llm | JsonOutputParser()

    # Step 3 — Invoke the chain. Pass:
    #   context = retrieved_docs[0].page_content[:500]  (first 500 chars of top chunk)
    #   query   = query
    #   answer  = answer
    context_excerpt = retrieved_docs[0].page_content[:500] if retrieved_docs else "(none)"

    result = judge_chain.invoke({
        "context": context_excerpt,
        "query": query,
        "answer": answer
    })
    #
    # Step 4 — Use a try/except so a bad LLM response doesn't crash the whole experiment:
    try:
        return result
    except (_json.JSONDecodeError, KeyError):
        print(f"[LLM-judge] WARNING: could not parse response, defaulting to []]. Raw: {result}")
        return []

## Preliminary Setup

In [ ]:
print("Loading documents...")
raw_documents = load_documents()
print(f"Loaded {len(raw_documents)} raw documents")

print("Preprocessing documents...")
processed_documents = preprocess_documents(raw_documents)
print(f"Created {len(processed_documents)} document chunks")

# RAG retriever
retriever = create_vector_store(processed_documents, embedding_model)

Loading documents...
Loaded 3 raw documents
Preprocessing documents...
Created 4359 document chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Experiment 1: RAG Retrieval

In [ ]:
# 1: RAG  (provided — do not modify)
# ----------------------------
def experiment_1():
    """ Run the complete RAG experiment."""
    print ("\n------- RAG Retrieval ---------")
    rag_results = []
    for i, query in enumerate(test_queries):
        print(f"\nQuery {i+1}: {query}")
        answer, retrieved_docs = run_rag_query(query, retriever)
        rag_results.append({"query": query, "answer": answer, "retrieved_docs": retrieved_docs})
        print(f"Answer: {answer}")
        time.sleep(2)  # avoid OpenAI rate limits

    return rag_results # return the results

In [ ]:
rag_results = experiment_1()
#print (rag_results) # for debugging -- show query, answer and returned responses


------- RAG Retrieval ---------

Query 1: What are the six classes of nutrients required for the human body to function properly?
Answer: The six classes of nutrients required for the human body to function properly are carbohydrates, lipids, proteins, water, vitamins, and minerals.

Query 2: Does water provide calories like carbohydrates, proteins, and lipids do?
Answer: I don't know based on the provided context.

Query 3: Why are nutrient-dense foods considered healthier than “empty-calorie” foods?
Answer: Nutrient-dense foods are considered healthier than “empty-calorie” foods because they contain many essential nutrients per calorie, providing significant amounts of one or more essential nutrients relative to the calories they provide. In contrast, empty-calorie foods, such as sugary carbonated beverages, provide many calories with very little, if any, other nutrients. Choosing more nutrient-dense foods can facilitate weight loss and contribute to overall better health.

Query 4:

## Experiment 2: LLM as Judge

In [ ]:
def experiment_2():
    """ Run the judge_llm on the retrieved results."""
    print ("\n------- LLM as Judge ---------")

    model_results = {
        "retrieval_scores": [],
        "answer_scores": [],
    }

    for i, r in enumerate(rag_results):
        scores = evaluate_results(r["query"], r["retrieved_docs"], r["answer"]) #LLM as judge

        ret_score = scores["retrieval_score"]
        ans_score = scores["answer_score"]
        model_results['retrieval_scores'].append(ret_score)
        model_results['answer_scores'].append(ans_score)
        print(f"\nQuery {i+1}:")
        print (f"- Retrieval Score: {ret_score}, Answer Score: {ans_score}\n- Reason: {scores['reason']}")
        time.sleep(2)

    model_results['avg_retrieval'] = np.mean(model_results['retrieval_scores'])
    model_results['avg_answer'] = np.mean(model_results['answer_scores'])

    return model_results

In [ ]:
judge_results = experiment_2()

print(f"\n** Avg Retrieval: {judge_results['avg_retrieval']:.2f}, "
      f"Avg Answer: {judge_results['avg_answer']:.2f}")


------- LLM as Judge ---------

Query 1:
- Retrieval Score: 5, Answer Score: 5
- Reason: The retrieved context is highly relevant as it directly lists the six classes of nutrients, and the answer correctly and completely reflects this information.

Query 2:
- Retrieval Score: 2, Answer Score: 5
- Reason: The retrieved context is minimally relevant as it does not address whether water provides calories, but the answer is correct because water does not provide calories like carbohydrates, proteins, and lipids do.

Query 3:
- Retrieval Score: 5, Answer Score: 5
- Reason: The retrieved context is highly relevant and the answer is correct and complete, explaining why nutrient-dense foods are healthier than empty-calorie foods.

Query 4:
- Retrieval Score: 5, Answer Score: 4
- Reason: The retrieved context is highly relevant as it directly addresses the role of physical activity in reducing disease risk and improving health, and the answer is mostly correct but includes additional details n

## Experiment 3: Temperature Experiment

In [ ]:
# First, create another rag_llm with a high temperature
rag_llm2 = ChatOpenAI(model="gpt-4o-mini", temperature=1.0)

# Pick one test_query for which the model found relevant chunks in the vector database
# (i.e., NOT 'Answer: I don't know based on the provided context.").
query = test_queries[2] # query for which the model correctly detected that it is answerable.

# Retrieve relevant chunks
retrieved_docs = retriever.invoke(query)

# Create a prompt for this experiment
prompt = PromptTemplate.from_template(
    """
    You are a helpful AI assistant answering questions about nutrition and diabetes.

    Use ONLY the information contained in the context below to answer the question.
    If the answer cannot be found in the context, clearly say something like
    "I don't know based on the provided context."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)

# Build a chain
chain = (
    {
        "context": lambda x: format_docs(retrieved_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | rag_llm2
    | StrOutputParser()
)

# Invoke the chain 3 times and collect answers
results = []
for _ in range(3):
    answer = chain.invoke(query)
    print(answer)
    results.append(answer)

#-----------------------
# For each answer, run the LLM judge to evaluate the medical correctness
# and Language simplicity.
#
# TODO (9): You write the rest

Nutrient-dense foods are considered healthier than “empty-calorie” foods because they contain many essential nutrients relative to the number of calories they provide. This means they provide significant amounts of vitamins, minerals, and other beneficial compounds that are important for health, while “empty-calorie” foods provide many calories but very few, if any, nutrients. Choosing nutrient-dense foods can facilitate weight loss and improve overall health.
Nutrient-dense foods are considered healthier than “empty-calorie” foods because they contain many essential nutrients relative to the amount of calories they provide. This means that nutrient-dense foods offer significant health benefits while helping to facilitate weight loss, as opposed to “empty-calorie” foods, which provide many calories with very little or no other nutrients.
Nutrient-dense foods are considered healthier than “empty-calorie” foods because they contain many essential nutrients per calorie, while empty-calori